# 01 · Hypotheses

**Research question:** how do the social and economic conditions of a country relate to the performance of the companies that operate there?

The first answer (notebook 03) was a weak association. This notebook sets out **all the hypotheses we test around that question**, how each one is tested, and what the data says. Each hypothesis has its own notebook; this one is the register and the scoreboard.

**How the tests are organised**
- Every hypothesis has **one primary test, defined before the result is looked at**, with a pre-agreed rule for the verdict (so a result cannot be re-interpreted afterwards).
- Company-level tests use controls for sector, relative size and fiscal year, **errors clustered by country** and a **wild cluster bootstrap** p-value (only 34 countries, so the usual p-values are too optimistic).
- Where possible the result is **replicated on ESEF**, an independent source (partly overlapping companies).
- All primary p-values are **corrected for multiple testing together** here (Benjamini–Hochberg).
- Everything is **association, not causation**.

**Run order:** each hypothesis notebook saves its result to `results/H*.json` when it runs. Run `03_analysis` and `1.1` to `1.16` first (`1.8` and `1.10` need `src/fetch/prices.py`, `1.16` needs `src/fetch/extra_indicators.py`); this notebook then reads them.

## 1 · The hypotheses

| ID | Hypothesis | Origin | Predicted | Primary test | Notebook |
|---|---|---|---|---|---|
| **H0** | Company ROA is related to the social and economic conditions of the country | The project's original question | Any association | Development index → ROA, with sector, size and year controls | `03_analysis` |
| **H1** | Where institutions are stronger, **fewer** listed companies make a loss | Suggested by an exploratory scan | Negative | Institutions index → probability of a loss | `1.1_risk_and_stability` |
| **H2** | Companies in **poorer** countries grow real revenue faster (catch-up) | Suggested by an exploratory scan | Negative | Development index → real revenue growth | `1.2_growth_and_catch_up` |
| **H3** | The development–ROA link is **explained by composition** (sector, size, leverage) | Follow-up to H0 | ≥ 50% of the link explained | Share of the link removed, country bootstrap | `1.3_mechanism_of_the_roa_link` |
| **H4** | The best sector **depends on the type of country** | Follow-up to the sector table | Interaction present | Permutation test of sector × type | `1.4_sector_by_country_type` |
| **H5** | **Deeper credit** goes with a **deeper stock market** (and institutions with new businesses) | Suggested by exploratory correlations | Positive | Spearman ρ, credit vs market value | `1.5_business_environment` |
| **H6** | In the EU, **higher labour cost** goes with **higher productivity** | Theory (sanity anchor) | Positive | Spearman ρ, labour cost vs productivity | `1.6_eu_labour_cost_and_productivity` |
| **H7** | Country features **improve predictions** of ROA for countries not seen in training | Methodological | Positive | Leave-one-country-out MAE gain | `1.7_out_of_country_prediction` |
| **H8** | Listed firms in more developed countries earn **lower real stock returns** | Extension of H0 to market outcomes | Negative | Development index → real calendar-year return | `1.8_market_outcomes` |
| **H9** | In years of stronger national growth, **the same firm** earns higher ROA | Within-company version of H0 | Positive | Company-demeaned ROA on company-demeaned GDP growth | `1.9_within_country_changes` |
| **H10** | Stronger institutions **cushioned the COVID crash** (Jan–Mar 2020) | Shock episode | Positive | Institutions index → crash return | `1.10_shocks_covid_and_energy` |
| **H11** | The H0 link **survives dropping any single country** and alternative definitions | Robustness of H0 (not a new claim) | Negative, stable | Weakest leave-one-country-out result | `1.11_sensitivity_of_the_main_finding` |
| **H12** | Firms in more developed countries **carry more debt** (lower equity ratio) | Financing outcome | Negative | Development index → equity ratio (non-financials) | `1.12_balance_sheets_and_financing` |
| **H13** | The development–ROA link is **weaker for large firms** | Follow-up to H0 | Positive interaction | Development × size rank interaction | `1.13_size_and_sector_splits` |
| **H14** | The development–ROA gap **narrows over time** (convergence) | Growth theory | Positive interaction | Development × time interaction | `1.14_convergence` |
| **H15** | **Yahoo and ESEF agree** on the same company-year ROA | Data-quality check (not a new claim) | Strong agreement | Spearman ρ on matched company-years | `1.15_source_check_yahoo_vs_esef` |
| **H16** | New **energy, climate, tax, banking and currency** indicators explain ROA | New data | Any association | Strongest of 12 indicators, BH corrected | `1.16_new_country_indicators` |

**A warning about origin.** H1, H2, H3 and H5 were suggested by looking at the same data, and H13 grew out of the H0 pattern, so their raw p-values flatter them (we looked, then tested). That is why each is checked in several samples and on ESEF where possible, and why the substantive tests are corrected together below. **Two entries are not new claims:** H11 re-tests H0 for fragility and H15 checks the data sources. They are reported in the scoreboard but left **out of the multiple-testing family**, because their p-values do not measure evidence for a new idea (H11 would count H0 twice).

In [1]:
import sys, pathlib, json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import utils
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_colwidth", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

reg = utils.load_results()
ORDER = [f"H{i}" for i in range(17)]
NOT_A_TEST = ["H11", "H15"]            # robustness / data-quality entries: shown, but outside the multiple-testing family
missing = [h for h in ORDER if h not in reg]
print("Results found for:", [h for h in ORDER if h in reg])
if missing:
    print("NOT RUN YET:", missing, "(run the corresponding notebooks)")

Results found for: ['H0', 'H1', 'H2', 'H3', 'H4', 'H5', 'H6', 'H7', 'H8', 'H9', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16']


## 2 · Scoreboard
One row per hypothesis. *Effect* is in the unit shown (they differ across hypotheses, so effects are not comparable). *p* is the primary test's p-value (method shown); *q* is the Benjamini–Hochberg value across all primary tests together. *Robust* is the share of robustness samples with the predicted sign. *Replication* is the independent check (ESEF, or an alternative measure).

In [2]:
rows = []
for h in ORDER:
    r = reg.get(h)
    if not r:
        rows.append({"id": h, "hypothesis": "(not run)", "verdict": "not run", "p": np.nan})
        continue
    rep = r.get("replication_coef")
    rows.append({"id": h, "hypothesis": r["title"], "effect": r["effect"], "unit": r["effect_unit"], "ci_low": r.get("ci_low"), "ci_high": r.get("ci_high"),
                 "p": r["p_primary"], "p method": r["p_method"], "n (countries)": r["clusters"], "robust": r["robust_share"],
                 "replication": rep, "replication p": r.get("replication_p"), "replication same sign": r.get("replication_same_sign"), "verdict": r["verdict"]})
sb = pd.DataFrame(rows).set_index("id")
fam = sb.drop(index=NOT_A_TEST, errors="ignore")
sb["q (BH)"] = utils.bh(fam.p).reindex(sb.index)
def after(row):
    v, q = row.verdict, row["q (BH)"]
    if row.name in NOT_A_TEST:
        return "not part of the test family"
    if v in ("supported", "partly supported"):
        return "holds after correction" if q < 0.10 else "does NOT survive correction"
    if v == "contradicted":
        extra = " (explained away by openness and size)" if reg.get(row.name, {}).get("explained_away") else ""
        return ("significant, opposite to prediction" if q < 0.10 else "opposite sign, not robust to correction") + extra
    return "-"
sb["after multiple-testing correction"] = sb.apply(after, axis=1)
COLORS = {"supported": "#c8e6c9", "partly supported": "#fff3b0", "not supported": "#e0e0e0", "contradicted": "#ffcdd2", "inconclusive": "#ffffff", "not run": "#ffffff"}
show = sb[["hypothesis", "effect", "unit", "p", "p method", "q (BH)", "n (countries)", "robust", "replication", "replication p", "verdict", "after multiple-testing correction"]]
(show.style.format({"effect": "{:+.2f}", "p": "{:.4f}", "q (BH)": "{:.3f}", "robust": "{:.0%}", "replication": "{:+.2f}", "replication p": "{:.3f}", "n (countries)": "{:.0f}"}, na_rep="")
     .map(lambda v: f"background-color: {COLORS.get(v, '')}", subset=["verdict"]))

,hypothesis,effect,unit,p,p method,q (BH),n (countries),robust,replication,replication p,verdict,after multiple-testing correction
id,,,,,,,,,,,,
H0,Country conditions and firm performance,-1.12,pp ROA per +1 SD development index,0.0010,wild cluster bootstrap,0.005,34,100%,-1.29,0.023,supported,holds after correction
H1,Risk and stability,+3.11,pp loss probability per +1 SD institutions,0.0135,wild cluster bootstrap,0.041,34,0%,+4.85,0.013,contradicted,"significant, opposite to prediction (explained away by openness and size)"
H2,Growth and catch-up,-1.41,pp real revenue growth per +1 SD development,0.1450,wild cluster bootstrap,0.362,34,100%,-2.90,0.220,not supported,-
H3,Mechanism of the development–ROA link,-0.00,share of the development–ROA link explained,0.5000,country bootstrap (share ≤ 0),0.734,34,0%,-0.13,0.021,not supported,-
H4,Sector × country type,+0.01,extra adjusted R² from sector × type interactions,0.2075,permutation of type labels across countries,0.389,34,0%,,,not supported,-
H5,Business environment,+0.77,"Spearman ρ, private credit vs stock-market value (% GDP)",0.0001,Spearman test (analytic),0.001,20,100%,-0.31,0.173,supported,holds after correction
H6,EU labour cost and productivity,+0.90,"Spearman ρ, hourly labour cost vs value added per employee",0.0000,Spearman test (analytic),0.000,21,96%,+0.88,0.000,supported,holds after correction
H7,Out-of-country prediction,-0.38,"MAE gain in pp ROA (baseline − full), leave-one-country-out",1.0000,bootstrap over countries (gain ≤ 0),1.000,34,40%,-1.21,1.000,not supported,-
H8,Stock returns and country development,-0.79,pp real stock return per +1 SD development,0.5870,wild cluster bootstrap,0.734,34,100%,-2.18,0.275,not supported,-


In [3]:
d = sb.dropna(subset=["p"]).copy()
d["-log10 p"] = -np.log10(d.p.clip(lower=1e-6))
fig = px.bar(d.reset_index(), x="id", y="-log10 p", color="verdict", color_discrete_map={"supported": "#4caf50", "partly supported": "#fbc02d", "not supported": "#9e9e9e", "contradicted": "#e53935"},
             hover_data=["hypothesis", "p", "q (BH)"], title="Strength of evidence for each hypothesis (higher = smaller p-value)")
fig.add_hline(y=-np.log10(0.05), line_dash="dot", line_color="black", annotation_text="p = 0.05")
fig.update_layout(height=420, xaxis_title="", yaxis_title="−log10(p), primary test"); fig.show()
n_tests = int(sb["q (BH)"].notna().sum())
print(f"{n_tests} primary tests in the family at α = 0.05: about {0.05 * n_tests:.1f} would pass by chance alone. Passing raw p < 0.05: {int((sb.loc[sb['q (BH)'].notna(), 'p'] < 0.05).sum())}; passing q < 0.10 after correction: {int((sb['q (BH)'] < 0.10).sum())}.")

15 primary tests in the family at α = 0.05: about 0.8 would pass by chance alone. Passing raw p < 0.05: 5; passing q < 0.10 after correction: 5.


## 3 · Did the replications agree?
An independent check for each hypothesis where one exists. Two different questions are answered separately: does the replication point the **same way as the main result**, and does it agree with the **prediction**? (For H1 the prediction was negative; both the main result and the ESEF replication are positive, so they agree with each other but not with the prediction.) The *kind* column says what the replication actually is, because they are not all the same test.

In [4]:
KIND = {"H0": "same test on ESEF", "H1": "same test on ESEF", "H2": "same test on ESEF", "H3": "ESEF: share removed by size + leverage only (no sector); the p shown is for the ESEF development–ROA link itself",
        "H4": "none possible (ESEF has no sector)", "H5": "different quantity: EU business births vs institutions (covers only the secondary half)",
        "H6": "alternative output measure (World Bank GDP per person)", "H7": "same test on ESEF",
        "H8": "different years of the same firms (2023–2024); ESEF has no prices", "H9": "same test on ESEF (2019–2025)", "H10": "ESEF accounts: change in ROA 2019→2020 (different outcome)",
        "H11": "leave-one-country-out on ESEF", "H12": "same test on ESEF", "H13": "same test on ESEF", "H14": "same test on ESEF 2018–2025",
        "H15": "the check itself; replication column = development–ROA coefficient on matched ESEF companies", "H16": "same indicator on ESEF EU country medians"}
rep = sb[["hypothesis", "effect", "replication", "replication p", "replication same sign"]].copy()
rep["kind of replication"] = rep.index.map(KIND)
comparable = ~rep.index.isin(["H4", "H5", "H8", "H10", "H11", "H15", "H16"])
rep["agrees with the main result"] = np.where(~comparable, "n/a", np.where(np.sign(rep.effect) == np.sign(rep.replication), "yes", "NO"))
rep["agrees with the prediction"] = np.where(rep["replication same sign"].isna(), "n/a", np.where(rep["replication same sign"].astype(bool), "yes", "no"))
rep.drop(columns="replication same sign")

,hypothesis,effect,replication,replication p,kind of replication,agrees with the main result,agrees with the prediction
id,,,,,,,
H0,Country conditions and firm performance,-1.124,-1.286,0.023,same test on ESEF,yes,yes
H1,Risk and stability,3.114,4.848,0.013,same test on ESEF,yes,no
H2,Growth and catch-up,-1.409,-2.903,0.220,same test on ESEF,yes,yes
H3,Mechanism of the development–ROA link,-0.002,-0.127,0.021,ESEF: share removed by size + leverage only (no sector); the p shown is for the ESEF development–ROA link itself,yes,no
H4,Sector × country type,0.014,NaN,NaN,none possible (ESEF has no sector),n/a,n/a
H5,Business environment,0.768,-0.309,0.173,different quantity: EU business births vs institutions (covers only the secondary half),n/a,no
H6,EU labour cost and productivity,0.903,0.884,0.000,alternative output measure (World Bank GDP per person),yes,yes
H7,Out-of-country prediction,-0.380,-1.215,1.000,same test on ESEF,yes,no
H8,Stock returns and country development,-0.793,-2.178,0.275,different years of the same firms (2023–2024); ESEF has no prices,n/a,yes


## 4 · What each notebook found
The key lines saved by each notebook, in order.

In [5]:
for h in ORDER:
    r = reg.get(h)
    if not r:
        continue
    display(Markdown(f"**{h} · {r['title']}** (`{r['notebook']}`) → **{r['verdict'].upper()}**\n\n" + "\n".join(f"- {n}" for n in r["notes"])))

**H0 · Country conditions and firm performance** (`03_analysis.ipynb`) → **SUPPORTED**

- Primary: +1 SD development → -1.12 pp ROA (95% CI -1.60 to -0.65), wild p = 0.001, 34 countries.
- Robustness: same sign in 100% of seven samples. ESEF replication: -1.29 pp (wild p = 0.023).
- VERDICT H0: SUPPORTED (association present, direction negative; effect small relative to firm and sector differences)

**H1 · Risk and stability** (`1.1_risk_and_stability.ipynb`) → **CONTRADICTED**

- Primary: +1 SD institutions → +3.11 pp probability of a loss (base rate 14.6%), wild p = 0.013, 34 countries.
- Robustness: expected sign in 0% of seven samples. Replication on ESEF: +4.85 pp (wild p = 0.013), opposite direction.
- Beyond openness and size: institutions coefficient +0.12 (p = 0.93) at country level when trade openness and population are controlled.
- VERDICT H1: CONTRADICTED (the association is explained away by trade openness and population)

**H2 · Growth and catch-up** (`1.2_growth_and_catch_up.ipynb`) → **NOT SUPPORTED**

- Primary: +1 SD development → -1.41 pp real revenue growth (typical 1.8%), wild p = 0.145, 34 countries, 5,039 company-years.
- Robustness: expected sign in 100% of seven samples. ESEF replication: -2.90 pp (wild p = 0.220), same direction.
- Controlling for national GDP growth the coefficient is -0.31 (p = 0.639).
- Profit vs growth at country level: ρ = +0.14 (n = 34).
- VERDICT H2: NOT SUPPORTED

**H3 · Mechanism of the development–ROA link** (`1.3_mechanism_of_the_roa_link.ipynb`) → **NOT SUPPORTED**

- Starting link: +1 SD development → -1.12 pp ROA (wild p = 0.001).
- Sector + size + leverage remove 0% of it (95% interval -36% to 20%, bootstrap p = 0.500). Sector alone: 0%; adding size: -1%; adding leverage: 0%.
- Channels (with controls): margin -3.01 pp (p = 0.000), asset turnover -0.01 (p = 0.628), equity ratio -0.000 (p = 0.990).
- Within sectors the link is negative in 10 of 11 sectors. ESEF: share removed by size + leverage -13%.
- What the data point to instead: the link works through thinner NET MARGINS within sectors, not through asset use or leverage.
- VERDICT H3: NOT SUPPORTED

**H4 · Sector × country type** (`1.4_sector_by_country_type.ipynb`) → **NOT SUPPORTED**

- Primary: sector × type interaction F = 6.20, extra adjusted R² = +0.0145, permutation p = 0.2075 (4 types, 34 countries).
- Robustness: significant in 0% of 6 variants.
- Largest departures (pp ROA, vs sector and type averages): Energy / type 1 -2.1, Consumer Defensive / type 4 +1.9.
- Not replicable on ESEF (no sector field).
- VERDICT H4: NOT SUPPORTED

...

**H12 · Financing: equity ratio and development** (`1.12_balance_sheets_and_financing.ipynb`) → **NOT SUPPORTED**

- Primary: +1 SD development → -0.29 pp equity ratio (typical 44%), wild p = 0.702, 34 countries, 9,051 company-years.
- Robustness: predicted sign in 100% of seven samples. ESEF replication: +0.69 pp (wild p = 0.838), opposite direction.
- Outcomes with q < 0.10 among the six: net margin (pp) (-3.64), operating margin (pp) (-3.55), return on equity (pp) (-2.41), loss-making firm-years (%) (+3.40).
- VERDICT H12: NOT SUPPORTED

**H13 · Size and sector: who carries the country effect** (`1.13_size_and_sector_splits.ipynb`) → **SUPPORTED**

- Primary: development × size interaction = +3.10 pp (effect at average size -1.12 pp per SD), wild p = 0.007, 34 countries, 10,220 company-years.
- By size: small -2.01; medium -1.23; large -0.19 pp per SD.
- Robustness: predicted sign in 100% of seven samples. ESEF replication: +6.06 pp (wild p = 0.004), same direction.
- By sector: development effect negative in 10 of 11 sectors; q < 0.10 in 5. Institutions × size: +2.98 (p = 0.008).
- VERDICT H13: SUPPORTED

**H14 · Convergence of company profitability** (`1.14_convergence.ipynb`) → **NOT SUPPORTED**

- Primary: the development slope changes by -0.05 pp per +1 SD per year (from -1.00 in 2022 to -1.22 in 2025), wild p = 0.471, 34 countries, 10,129 company-years.
- Robustness: predicted sign in 0% of seven samples. ESEF 2018–2025: +0.06 pp per year (wild p = 0.893), same direction.
- σ-convergence: SD of country median ROA 1.36 in 2022 vs 1.29 in 2025.
- β-convergence of GDP per person 2012–2024: slope -1.01 (ρ = -0.32, permutation p = 0.075, n = 34); EU only ρ = -0.68 (p = 0.002).
- VERDICT H14: NOT SUPPORTED

**H15 · Source agreement: Yahoo vs ESEF** (`1.15_source_check_yahoo_vs_esef.ipynb`) → **SUPPORTED**

- Primary: Spearman ρ = +0.992 between Yahoo and ESEF ROA over 1,471 matched company-years (484 companies, 18 countries); median |difference| 0.00 pp, 97% within 1 pp.
- Countries (≥ 8 pairs): ρ ≥ 0.7 in 100% of 15; lowest +0.93 (AUT). Ingredient ratios ESEF/Yahoo (median): net income 1.00, assets 1.00, revenue 1.00.
- Gap vs development: -0.17 pp per SD (wild p = 0.245). The development–ROA coefficient on the same companies: Yahoo +0.23 (p = 0.820), ESEF +0.17 (p = 0.874).
- Implication: agreement this close means Yahoo is, for these companies, effectively reproducing the filed accounts, so ESEF replications are NOT independent for the 484 overlapping companies (37% of ESEF companies). They are independent only for the rest.
- VERDICT H15: SUPPORTED

**H16 · New indicators: energy, climate, tax, banking, currency** (`1.16_new_country_indicators.ipynb`) → **NOT SUPPORTED**

- Primary: strongest of 12 new indicators is 'broadband per 100 people': ρ = -0.40 with median ROA (raw p = 0.0191, BH q = 0.177, n = 34 countries); 0 indicators pass q < 0.10.
- Given the development index the partial ρ is -0.26; firm-level coefficient -0.97 pp per SD (wild p = 0.007), -0.36 pp with development controlled (p = 0.191).
- Robustness: same sign in 100% of seven samples. ESEF (EU only) ρ = -0.18 (p = 0.480), same direction.
- Prediction (leave-one-country-out MAE): constant 0.91, development 0.91, new indicators 1.10, both 1.12 pp.
- VERDICT H16: NOT SUPPORTED

## 5 · Reading across the results
Written from the results of the last full run (6 October 2026; points 1–5 from H0 to H7, point 6 for H8 to H16). The next cell checks that the stored verdicts still match; if anything changed, revisit this section.

In [6]:
expected = {"H0": "supported", "H1": "contradicted", "H2": "not supported", "H3": "not supported", "H4": "not supported", "H5": "supported", "H6": "supported", "H7": "not supported",
            "H8": "not supported", "H9": "not supported", "H10": "not supported", "H11": "supported", "H12": "not supported", "H13": "supported", "H14": "not supported", "H15": "supported", "H16": "not supported"}
got = {h: reg[h]["verdict"] for h in ORDER if h in reg}
diff = {h: (expected[h], got.get(h)) for h in expected if got.get(h) != expected[h]}
print("Verdicts match the interpretation below." if not diff else f"⚠ RESULTS CHANGED since the interpretation was written: {diff}. Revisit section 5.")

Verdicts match the interpretation below.


**1. Country conditions are related to company profitability, but weakly (H0, H7).** Firms in more developed countries earn about 1 percentage point less ROA per standard deviation of development, in 7 of 7 samples and on the independent ESEF source. But this is a small slice of the variation (firm identity dominates), and country features **do not improve predictions** for a country the model has not seen (H7): with only 34 countries, adding the features makes predictions worse, not better.

**2. The link is not an artefact of which companies are listed (H3).** Sector mix, size and leverage explain none of it. Within sectors the link is negative in 10 of 11, and it works through **thinner net margins**, not through how intensively assets are used or how firms are financed.

**3. Two plausible stories did not hold.** *Risk (H1):* stronger institutions go with *more* loss-making firms, opposite to the prediction, but that is explained by trade openness and country size (small open economies), not by institutions. *Catch-up (H2):* poorer-country firms grow faster in nominal terms, but after removing inflation the gap is not distinguishable from zero, and it mostly reflects those countries' faster national growth. *Sector × country type (H4):* no evidence that sector differences depend on the type of country.

**4. The clearer relationships are at country level, around the firms, not in their profits.** Deeper credit systems go with deeper stock markets (H5, ρ = +0.77, stable across time windows), and in the EU labour cost tracks productivity almost one-for-one (H6, elasticity about 0.9). Institutions and new-business formation did not show a clear link (H5 secondary, EU replication).

**5. Overall message for the report.** *Country conditions go with the economic environment of firms (finance, wages, productivity) far more clearly than with how profitable the firms are.* Profitability depends mainly on the company; the small country effect that exists shows up as lower margins in more developed countries and cannot be turned into predictions.

**6. Second round: nine more questions (H8 to H16).**
- **The country effect lives in small firms (H13, supported).** The development–ROA link is −2.0 pp per SD for the smallest third of firms in each country, −1.2 for the middle and −0.2 (about nothing) for the largest third; the interaction is +3.1 pp (corrected q = 0.03) and holds in every robustness sample and on ESEF. This refines point 1: the weak average hides a real effect concentrated in firms that depend on the domestic economy.
- **H0 is not fragile (H11, supported).** Dropping any one of the 34 countries keeps the coefficient negative and significant (weakest case: dropping India, −0.9 pp), as does dropping the five most influential, 9 of 9 alternative definitions and a country-shuffling placebo. It stays small, however: this tests stability, not importance.
- **Several natural extensions found nothing (not supported):** stock returns (H8) do not follow the ROA pattern, though volatility is lower in more developed countries; a firm's own ROA does not move with national growth within a firm over time (H9: the between-country slope is positive, the within-firm one is zero, so the cross-country pattern is not a response to growth); stronger institutions did not cushion the early-2020 crash (H10); financing structure does not vary with development (H12) even though margins do (net margin −3.6 pp per SD); no convergence of company profitability over 2022–2025 (H14, though GDP per person does converge, ρ = −0.32, EU −0.68); and none of 12 new energy, climate, tax, banking or currency indicators survive correction (H16).
- **Data quality is good, and the replications are less independent than we said (H15).** Yahoo and ESEF give essentially the same ROA for the same company (ρ = 0.99, 97% within 1 pp), so for the 484 overlapping companies (37% of ESEF) the ESEF replications are not independent. The agreement also means Yahoo's numbers are not a source of the country patterns.

**Multiple testing.** The test family is H0 to H10, H12, H13, H14 and H16 (15 tests; H11 and H15 are excluded because they re-test H0 and check the data rather than add a claim). About 0.8 would pass at 5% by chance; 5 pass after correction (q < 0.10): H0, H1, H5, H6 and H13. H1, H2, H3, H5 and H13 were suggested by looking at the data, so they should be presented as exploratory; H6 is the one result that is a sanity anchor rather than a finding about firms.

## 6 · Limits that apply to every hypothesis
- **34 countries** is the effective sample for anything about countries; thousands of firm-years do not change that.
- **Listed, large companies** only, headquarters country (not where the profit is made), about four fiscal years, mixed accounting standards.
- **Overlapping sources:** Yahoo and ESEF share some companies, so the ESEF replications are partly, not fully, independent.
- **Overlapping features:** the country features are strongly correlated (mostly one development factor), so separate effects of wealth, institutions and openness cannot be cleanly told apart.
- **Association, not causation.**